# ???????????? ??????? (?????)
link: https://ai.stackexchange.com/questions/13254/what-are-options-in-reinforcement-learning

????? ??????? ????? ???????? ?????? ?????, ?????? ?? ??????? ?????? ???? ??????? ????????? ???????????? ????????? ? ?????? ?????. ??? ???????? ?????? ????????????? ??????????? ????????? Q-????????.

In [ ]:
try:
    import google.colab
    COLAB = True
except ModuleNotFoundError:
    COLAB = False
    pass

if COLAB:
    !pip -q install "gymnasium[classic-control, atari, accept-rom-license]"
    !pip -q install piglet
    !pip -q install imageio_ffmpeg
    !pip -q install moviepy==1.0.3

In [ ]:
# ??????????? ????? ? ??????? ?????????
import gymnasium as gym
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

environment = gym.make('Taxi-v3', render_mode='ansi')
environment.reset()
print(environment.render())

In [ ]:
class QLearningAgent:
    def __init__(self, learning_rate, epsilon, gamma, get_legal_actions, seed=1337):
        self.get_legal_actions = get_legal_actions
        self.Q = defaultdict(lambda: defaultdict(lambda: 0))  # when called, non-existent values appear as zeros
        self.learning_rate = learning_rate
        self.epsilon = epsilon
        self.gamma = gamma
        self.rng = np.random.default_rng(seed)
    
    def Qs(self, state, actions):
        return np.array([self.Q[state][action] for action in actions])

    def V(self, state):
        """Returns max_action Q(state,action) where the max is over legal actions."""
        possible_actions = self.get_legal_actions(state)
        if not possible_actions:
            return 0.

        return np.max(self.Qs(state, possible_actions))


    def act(self, state):
        """
        Compute the action to take for the current state, including exploration.
        With probability self.epsilon, we should take a random action,
        otherwise - the best policy action (self.getPolicy).
        """

        possible_actions = self.get_legal_actions(state)
        if not possible_actions:
            return None

        if self.rng.random() < self.epsilon:
            action = self.rng.choice(possible_actions)
        else:
            best_action_ind = np.argmax(self.Qs(state, possible_actions))
            action = possible_actions[best_action_ind]
        return action

    def update(self, state, action, next_state, reward, terminated):
        lr, gamma, r, s, a, next_s = self.learning_rate, self.gamma, reward, state, action, next_state
        not_terminated = int(not terminated)
        Q, V = self.Q, self.V
        
        td_error = r + gamma * not_terminated * V(next_s) - Q[s][a]
        Q[s][a] += lr * td_error

### ??????? 1 
?????????? ??? ??????????? ????? Taxi: https://github.com/openai/gym/blob/master/gym/envs/toy_text/taxi.py

???????? 4 ????????? ??????????? Taxi, ? ??????? ????? ?????? ????? ?????????? ????? ?? ?????: R, G, B, Y ??????????????. 

In [ ]:
class TaxiStepWrapper(gym.Wrapper):
    def __init__(self, env, target_id, target_reward):
        super().__init__(env)
        #self.locs  = RGBY position
        self._target = self.locs[target_id]
        self._target_reward = target_reward

    def target_reached(self, state) -> bool:
        taxi_pos_x, taxi_pos_y, p_idx, d_idx = list(self.env.decode(state))
        return (taxi_pos_x, taxi_pos_y) == self._target 

    def step(self, action):
        # ???????? ????????? (state, reward, _, obs),
        # ??????? ???????? ?????, ????????? ????? step 
        # ????????? ???????? ?? ????????? ???????????
        # ??? ?????? ????????????????? ?????????
        # ???????? ?????????????? (reward) ? 
        # ???? ?????????? ??????? (is_done)
        # ?? ?????? ???????? ????? ?????? ?????????????? 
        # -1, ?? ?????????? ???? - self._target_reward
        ####### ????? ??? ??? ########
        state, reward, terminated, truncated, info = super().step(action)
        taxi_pos_x, taxi_pos_y, p_idx, d_idx = list(self.decode(state))
        
        terminated = ((taxi_pos_x, taxi_pos_y) == self._target) 
        reward = self._target_reward if terminated else -1
        ##############################

        return state, reward, terminated, truncated


???????? ???? ??????? (wrapper), ????????? ????????? ?????????.  ??????? ????? ?????? ????  R, G, Y, B.

In [ ]:
for idx in range(4):
    # ??????? ????????? ? ???????? ??????? ??????????
    ####### ????? ??? ??? ########
    # print(environment.locs[idx])
    test_env = TaxiStepWrapper(env = environment, target_id = idx, target_reward = 100)
    terminated = False
    ret = 0
    ##############################
    
    # ????????? ????????? ?????????,
    # ???? ?????? ?? ??????????
    ####### ????? ??? ??? ########
    _, _ = test_env.reset()
    while not terminated:
         state, reward, terminated, truncated = test_env.step(test_env.action_space.sample())
         ret += reward
    ##############################

    print(test_env.render())
    print(f"state:{state} reward:{reward} ret:{ret}")

In [ ]:
def play_and_train(env, agent, t_max=10 ** 4):
    total_discounted_reward = 0.0
    s, _ = env.reset()
    for t in range(t_max):
        a = agent.act(s)
        try:
            next_s, r, terminated, truncated = env.step(a)
        except ValueError:
            next_s, r, terminated, truncated, _ = env.step(a)
        done  = terminated or truncated
        agent.update(s, a, next_s, r, done)
        s = next_s
        total_discounted_reward += r
        if done: break
    return total_discounted_reward

### ??????? 2 
1. ?????? ??????? ?? ????????? ???? ??????????.
2. ???????? ?????????? ??????? ?????, ?????? ????? ????? ????? ?????????, ????????? ????????? ????????? ? ????????? ???????? ?????????.

In [ ]:
n_actions = environment.action_space.n
seed = 1331
rng = np.random.default_rng(seed)

# ?????????, ??????? ????? ???????????? ??????
params = dict(
    learning_rate=0.1,
    epsilon=0.1, 
    gamma=0.99,
    get_legal_actions=lambda s: range(4),
    seed=rng.integers(100_000_000)
)

# ??????? ??????? 
agents_for_options = [
    QLearningAgent(**params) for _ in range(4)
]

for idx in range(4):
    # ??????? ????????? ? ???????? ??????? ??????????, 
    # ????????? ????????? ????????? ??????? ???????
    ####### ????? ??? ??? ########
    env = TaxiStepWrapper(env = environment, target_id = idx, target_reward = 100)
    print(
        f"total discounted reward for agent {idx}: {play_and_train(env, agents_for_options[idx])}"
    )
    ##############################

In [ ]:
# ????????? ????? ?????
class Option:
    def __init__(self, policy, termination_prob, initial, seed=None):
        self.policy = policy
        self.termination_prob = termination_prob
        self.initial_states = initial
        self.rng = np.random.default_rng(seed)

    def can_start(self, state):
        return state in self.initial_states

    def terminate(self, state):
        return self.rng.random() <= self.termination_prob[state]

    def act(self, state):
        return self.policy.act(state)

In [ ]:
options = []
for index, agent in enumerate(agents_for_options):
    # ??????? ??????? termination_prob, ? ??????? ??????? 
    # ?????????  ????? ?????? ??????????? ?????????? 
    # ?????. ? ????? ?????? ??????? 1.0 ??? 0.0, 
    # ? ??????????? ?? ?????????.
    # ??????? ????????? initial, ????????? ? ???? 
    # ?????????, ?? ??????? ????? ????? ???? 
    # ??????? (??? ????? ???????)
    termination_prob = dict()
    initial_states = set()
    termination_states = set()
    
    ####### ????? ??? ??? ########
    tmp_env = TaxiStepWrapper(env = environment, target_id = index, target_reward = 100)
    
    termination_prob = {
        s: 1.0 if tmp_env.target_reached(s) else 0.0 for s in range(environment.observation_space.n) 
    }
    initial_states = set(termination_prob.keys())

    ##############################    

    options.append(
        Option(
            policy=agent, 
            termination_prob=termination_prob,
            initial=initial_states
        )
    )


### ??????? 3
??????? ???????, ??????? ????? ????????? ????? ? ?????????? ???????????????? ??????????????, ???????? ?? ????? ??????????? ????????
$$ R = r_{1} + \gamma r_{2} + \gamma^{2} r_{3} + \dots + \gamma^{t-1}r_{T}$$

In [ ]:
def apply_option(option, gamma, env, debug=False):
    reward = 0
    steps = 0
    alpha = 1
    state = env.unwrapped.s
    if not option.can_start(state):
        raise KeyError
    
    # ??????????????? ?? ?????? ???? ????? ??? ????????? 
    # ?? ??????????, ??????? ???????????????? 
    # ?????????????? reward (?????????? steps),
    # ????? ??????? render ?????????, ??? ????? - debug
    
    ####### ????? ??? ??? ########
    while True:
        state, reward_t, terminated, truncated, info = env.step(option.act(state))
        terminated |=  option.terminate(state)
        if terminated or truncated:
            break

        reward += reward_t * alpha
        alpha *= gamma
        steps += 1

    if debug:
        print(env.render())
    ##############################

    return state, reward, terminated, truncated

In [ ]:
# ???????? ?????? ??????
env = environment
s, _ = env.reset()

r = apply_option(options[0], 0.99, env, debug=True)

???????, ??? ??? ??????, ?? ?? ?????? ??????????? ???????, ????? ???????? ????? ?????????? ? ?????! ????????? ????? ? ?????????, ??? ????????? ?? ??? ????????? ? ?????????, ??? ????? ????  ???? ?? ?????.

In [ ]:
s, _ = env.reset()
env.unwrapped.s = 499
print(env.render())
print("\n" * 2)
r = apply_option(options[0], 0.99, env, debug=True)
print(r[1])

### ??????? 4
?????, ??? ????? ?? ????????? ??????????? ? ????? ????????. 
???????? ???? ??????? ???????? ???, ????? ????? ???????? ????????? ??? ???? ????????? ????????? ????? ? ??????????? ?? ??????.

In [ ]:
def play_and_train_modified(env, agent, t_max=10 ** 4, rng=None):
    # ??????? ????? ??????? play_and_train, ??????? 
    # ? ???????? ?????????? ????????? ???????? ????? 
    # ????????? ?????,??????? ? ??, ????? ???????? 
    # ????????? ? ?????
    if rng is None or isinstance(rng, int):
        rng = np.random.default_rng(rng)

    total_discounted_reward = 0.0
    s, _ = env.reset()
    
    # ???????? ????????? ????????? ?????
    # (?????????? ????? env.uwrapped)
    ####### ????? ??? ??? ########
    s = env.unwrapped.observation_space.sample()
    ##############################

    for t in range(t_max):
        a = agent.act(s)
        next_s, r, terminated, truncated = env.step(a)
        done = terminated or truncated
        agent.update(s, a, next_s, r, done)
        s = next_s
        total_discounted_reward += r
        if done: break
            
    return total_discounted_reward


for index in range(4):
    for _ in range(7000):
        wrapped_env = TaxiStepWrapper(
            env=environment, 
            target_id=index, target_reward=50
        )
        play_and_train_modified(
            env=wrapped_env, 
            agent=agents_for_options[index],
            rng=rng
        )


???????? ?????? ?????? ????????? ??? ? ????????, ??? ????? ???????? ??? ???? ???????!

In [ ]:
env = environment
s, _ = env.reset()

env.unwrapped.s = rng.integers(0, 499)
r = apply_option(options[0], 0.99, env, debug=True)
print(r)

### ??????????? ? ????????
?????????? ????????, ????????? ???????????? (????? ?? ?????? ????????) ? ????????? ?????.

#### ??????? ???????????? ????? (????? ?? ?????? ????????: ??????? ? ??????? ?????????):

In [ ]:
# ??? ???????? 4-5 (pickup, dropoff) ??????? 
# ???????????? ?????:
class OneActionAgent:
    def __init__(self, action):
        self.action = action
    
    def act(self, state):
        return self.action
    
    def update(*args, **kwargs):
        pass

options = options[:4]    
for action in range(4, 6):
    # ???????????? ????? ?????????? ? ????? ?????????, 
    # ????????? ????? ???????? ? ???????????
    initial = set(range(environment.observation_space.n))
    termination_prob = {
        s: 1.0
        for s in range(environment.observation_space.n)
    }
    options.append(
        Option(policy=OneActionAgent(action), termination_prob=termination_prob, initial=initial)
    )

#### ???????? ???????????? ?????:

In [ ]:
env = environment
s, _ = env.reset()

env.unwrapped.s = rng.integers(0, 499)
_ = apply_option(options[0], 0.99, env, debug=True)
_ = apply_option(options[4], 0.99, env, debug=True)

#### ????????? ??????? ??? ?????????, ??????? ?????? ???????? ????????? ????? (? ???????? ????? - ?????? ?????):

In [ ]:
class OptionTaxiStepWrapper(gym.Wrapper):
    def __init__(self, env, options, gamma=0.99):
        self.options = options
        self.gamma = gamma
        super().__init__(env)

    def step(self, action):
        state, reward, terminated, truncated = apply_option(
            self.options[action], self.gamma, self.unwrapped
        )
        return state, reward, terminated, truncated


In [ ]:
baseline_params = dict(
    learning_rate=0.1,
    epsilon=0.1, 
    gamma=0.99,
    get_legal_actions=lambda s: range(environment.action_space.n),
    seed=rng.integers(100_000_000)
)
baseline_agent = QLearningAgent(**baseline_params)



In [ ]:
option_agent = QLearningAgent(
    learning_rate=0.1, epsilon=0.2, gamma=0.99,
    get_legal_actions=lambda s: range(len(options)),
    seed=rng.integers(100_000_000)
)
from IPython.display import clear_output
ema = [0]
alpha = 0.1
# ??????? ?????????, ???????????? ?????
env = OptionTaxiStepWrapper(gym.make('Taxi-v3', render_mode='ansi'), options=options)

rewards = []
baseline_rewards = []

for episode in range(5000):
    r = play_and_train(env=env, agent=option_agent)
    rewards.append(r)
    ema.append(alpha * r + (1 - alpha) * ema[-1])
    # baseline_rewards.append(
    #     play_and_train(env=env, agent=baseline_agent)
    # )

    if episode % 100 == 0:
        clear_output(True)
        option_agent.epsilon *= 0.94
        # baseline_agent.epsilon *= 0.94

        # plt.plot(baseline_rewards, label = "baseline")
        plt.plot(rewards, label = "with options")
        plt.plot(ema, label = r"EMA with $\alpha$=" + str(alpha))

        plt.legend(loc="best")
        plt.show()